In [20]:
# Imports
from typing import TypedDict, Generator, List
import json
from openai import OpenAI
import gradio as gr

In [22]:
# Configs

# LLM Client
openai = OpenAI()
OPEN_AI_MODEL = "gpt-4o-mini"

# LLM Instructions
system_message = "You are a helpful assistant for an Airline called FlightAI. \
    Give short, courteous answers, no more than 1 sentence. \
        Always be accurate. If you don't know the answer, say so."

# Tools
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [ ]:
class ChatMessage(TypedDict):
    role: str
    content: str


def get_ticket_price(destination: str):
    return ticket_prices.get(destination.lower(), "Unknown")


def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get("destination")
    price = get_ticket_price(city)
    response = {
        "role": "tool",
        "content": json.dumps({"destination": city, "price": price}),
        "tool_call_id": tool_call.id,
    }
    return response


def chat(message: str, history: List[ChatMessage]) -> str:
    messages = [{"role": "system", "content": system_message}]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": message})

    response = openai.chat.completions.create(
        model=OPEN_AI_MODEL, messages=messages, tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(
            model=OPEN_AI_MODEL, messages=messages
        )

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/queueing.py", line 626, in process_events
    response = await route_utils.call_process_api(
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/route_utils.py", line 350, in call_process_api
    output = await app.get_blocks().process_api(
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/blocks.py", line 2235, in process_api
    result = await self.call_function(
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/blocks.py", line 1744, in call_function
    prediction = await fn(*processed_input)
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/utils.py", line 884, in async_wrapper
    response = await f(*args, **kwargs)
  File "/home/alfred_sasko/llm_engineering/.venv/lib/python3.10/site-packages/gradio/chat_interface.py", line 55